In [9]:
# ==========================================================
# HOUSE PRICE PREDICTION USING XGBOOST + GRIDSEARCH + GRADIO
# ==========================================================

# Install libraries
!pip install -q kaggle xgboost gradio

# Upload kaggle.json
from google.colab import files
uploaded = files.upload()

# Configure Kaggle API
import os

os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Download Dataset
!kaggle competitions download -c house-prices-advanced-regression-techniques

# Extract Dataset
!unzip -o house-prices-advanced-regression-techniques.zip

# ==========================================================
# IMPORT LIBRARIES
# ==========================================================

import pandas as pd
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error

from xgboost import XGBRegressor

# ==========================================================
# LOAD DATASET
# ==========================================================

df = pd.read_csv("train.csv")

# ==========================================================
# SELECT IMPORTANT FEATURES
# ==========================================================

features = [
    'OverallQual',
    'GrLivArea',
    'GarageCars',
    'GarageArea',
    'TotalBsmtSF',
    '1stFlrSF',
    '2ndFlrSF',
    'FullBath',
    'TotRmsAbvGrd',
    'YearBuilt',
    'YearRemodAdd',
    'Fireplaces',
    'MasVnrArea',
    'LotArea',
    'BsmtFinSF1',
    'OverallCond',
    'GarageYrBlt',
    'WoodDeckSF',
    'OpenPorchSF',
    'BedroomAbvGr'
]

X = df[features]
y = df["SalePrice"]

# Fill missing values
X = X.fillna(X.mean())

# ==========================================================
# TRAIN TEST SPLIT
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================================================
# FINE TUNING
# ==========================================================

param_grid = {
    'n_estimators':[200,300],
    'max_depth':[4,5,6],
    'learning_rate':[0.03,0.05]
}

grid = GridSearchCV(
    estimator=XGBRegressor(
        objective='reg:squarederror',
        random_state=42
    ),
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

grid.fit(X_train,y_train)

best_model = grid.best_estimator_

print("\nBest Parameters")
print(grid.best_params_)

# ==========================================================
# EVALUATION
# ==========================================================

pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test,pred)

print("\nMean Absolute Error (MAE):",mae)

# ==========================================================
# PREDICTION FUNCTION
# ==========================================================

def predict_price(
    OverallQual,
    GrLivArea,
    GarageCars,
    GarageArea,
    TotalBsmtSF,
    FirstFlrSF,
    SecondFlrSF,
    FullBath,
    TotRmsAbvGrd,
    YearBuilt,
    YearRemodAdd,
    Fireplaces,
    MasVnrArea,
    LotArea,
    BsmtFinSF1,
    OverallCond,
    GarageYrBlt,
    WoodDeckSF,
    OpenPorchSF,
    BedroomAbvGr
):

    data = pd.DataFrame([[
        OverallQual,
        GrLivArea,
        GarageCars,
        GarageArea,
        TotalBsmtSF,
        FirstFlrSF,
        SecondFlrSF,
        FullBath,
        TotRmsAbvGrd,
        YearBuilt,
        YearRemodAdd,
        Fireplaces,
        MasVnrArea,
        LotArea,
        BsmtFinSF1,
        OverallCond,
        GarageYrBlt,
        WoodDeckSF,
        OpenPorchSF,
        BedroomAbvGr
    ]],columns=features)

    prediction = best_model.predict(data)[0]

    return f"Estimated House Price : ${prediction:,.2f}"

# ==========================================================
# UI
# ==========================================================

demo = gr.Interface(
    fn=predict_price,

    inputs=[
        gr.Slider(1,10,value=7,label="Overall Quality"),
        gr.Number(value=1800,label="Living Area"),
        gr.Number(value=2,label="Garage Cars"),
        gr.Number(value=500,label="Garage Area"),
        gr.Number(value=900,label="Basement Area"),
        gr.Number(value=1200,label="1st Floor Area"),
        gr.Number(value=600,label="2nd Floor Area"),
        gr.Number(value=2,label="Bathrooms"),
        gr.Number(value=7,label="Total Rooms"),
        gr.Number(value=2005,label="Year Built"),
        gr.Number(value=2015,label="Year Remodeled"),
        gr.Number(value=1,label="Fireplaces"),
        gr.Number(value=100,label="Masonry Area"),
        gr.Number(value=8000,label="Lot Area"),
        gr.Number(value=600,label="Finished Basement Area"),
        gr.Slider(1,10,value=6,label="Overall Condition"),
        gr.Number(value=2005,label="Garage Year Built"),
        gr.Number(value=120,label="Wood Deck Area"),
        gr.Number(value=40,label="Open Porch Area"),
        gr.Number(value=3,label="Bedrooms")
    ],

    outputs=gr.Textbox(label="Predicted House Price"),

    title="🏠 House Price Prediction using Machine Learning",

    description="Predict house prices using an XGBoost model trained on the Kaggle House Prices dataset."
)

demo.launch(share=True)

Saving kaggle.json to kaggle.json
100% 199k/199k [00:00<00:00, 358kB/s]

Archive:  house-prices-advanced-regression-techniques.zip
  inflating: data_description.txt    
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               

Best Parameters
{'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 300}

Mean Absolute Error (MAE): 17013.767578125
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://550a409a3d44807c41.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
